**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Filter Design

The application-focused companion to [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb): there we learned *what* the frequency domain is; here we learn to *sculpt* it. By the end you will design FIR and IIR filters, read their responses, and apply them to real signals.

## 0. Introduction

A filter is a system that treats different frequencies differently — keep the heartbeat, drop the power-line hum; keep the voice, drop the hiss. Digital filters come in two families with a classic trade-off:

| | FIR | IIR |
|---|---|---|
| Impulse response | finite | infinite (feedback) |
| Always stable? | **yes** | no — poles must stay in the unit circle |
| Exactly linear phase? | can be | no |
| Order needed for sharp cutoff | high | **low** |


## 1. Pre-requisites

- [Intro to Python](../Intro_Func_Prog/Intro_Python/Intro_Python.ipynb) — NumPy & Matplotlib.
- [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb) — convolution, DTFT/DFT, the $z$-plane (Sessions 2–3 especially).

We use `scipy.signal` throughout — install with `conda install scipy` or `pip install scipy`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 1000  # sampling rate [Hz] used throughout

def plot_response(b, a=1, fs=fs, title=""):
    """Magnitude (dB) and phase of a digital filter."""
    w, h = signal.freqz(b, a, worN=2048, fs=fs)
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
    ax0.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-8)))
    ax0.set_ylabel("magnitude [dB]"); ax0.set_ylim(-100, 5); ax0.grid(True)
    ax0.set_title(title)
    ax1.plot(w, np.unwrap(np.angle(h)))
    ax1.set_ylabel("phase [rad]"); ax1.set_xlabel("frequency [Hz]"); ax1.grid(True)
    plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *FIR Filters* (~35 min)
**Goal:** design a windowed-sinc FIR low-pass, read its response, and understand linear phase.
**Builds on:** [Foundations](./Foundations_of_Signal_Processing_1.ipynb) Sessions 3 & 6. &nbsp; **Feeds into:** Session 2 (IIR).

---

## 2. FIR Filters

💡 **Intuition.** An FIR filter is nothing but a **weighted moving average**: the output is a fixed set of weights (the *taps*) slid across the signal — a convolution. The ideal low-pass in frequency is a rectangle, whose time-domain shape is the infinite `sinc`; a practical FIR filter is that sinc *truncated and tapered by a window*. More taps ⇒ closer to the rectangle ⇒ sharper cutoff.

### 2.1. The Windowed-Sinc Design

`firwin` does exactly the recipe above: sample the ideal sinc, multiply by a window (Hamming by default), normalize.

In [2]:
numtaps = 101          # filter length (order + 1); odd keeps a symmetric center tap
cutoff = 100           # Hz

taps = signal.firwin(numtaps, cutoff, fs=fs)

plt.figure(figsize=(8, 2.5))
plt.stem(taps)
plt.title("The taps ARE a windowed sinc")
plt.xlabel("tap index"); plt.tight_layout(); plt.show()

/tmp/ipykernel_1835677/399986226.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("tap index"); plt.tight_layout(); plt.show()


In [3]:
plot_response(taps, title="FIR low-pass: 101 taps, 100 Hz cutoff")

/tmp/ipykernel_1835677/650736124.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Things to read off the plot:

- **Passband** (0–100 Hz): flat at 0 dB — frequencies pass untouched.
- **Transition band**: the rolloff around the cutoff; narrower needs more taps.
- **Stopband**: ripples bounded by the window's sidelobe level (Hamming ⇒ ~−53 dB).
- **Phase**: a perfectly straight line — *linear phase*.

💡 **Intuition.** Linear phase means **every frequency is delayed by the same amount** ($\frac{N-1}{2}$ samples — half the filter length). The waveform's shape survives; it just arrives late. Nonlinear phase smears different frequencies by different delays, *distorting the shape even when magnitudes are untouched* — fatal for waveforms you must interpret (ECG, seismic, communications symbols).

### 2.2. Windows Trade Ripple for Width

The window choice is a knob: low sidelobes (less stopband ripple) cost a wider mainlobe (slower rolloff) — the uncertainty principle from [Foundations Session 4](./Foundations_of_Signal_Processing_1.ipynb) wearing a hard hat.

In [4]:
plt.figure(figsize=(8, 3))
for win in ["boxcar", "hamming", "blackmanharris"]:
    t = signal.firwin(numtaps, cutoff, fs=fs, window=win)
    w, h = signal.freqz(t, worN=2048, fs=fs)
    plt.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-8)), label=win)
plt.ylim(-120, 5); plt.legend(); plt.grid(True)
plt.title("Same cutoff, three windows: sidelobes vs rolloff")
plt.xlabel("frequency [Hz]"); plt.ylabel("magnitude [dB]")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1835677/1539968725.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *IIR Filters* (~35 min)
**Goal:** design Butterworth/Chebyshev filters; understand stability from pole locations.
**Builds on:** Session 1; [Foundations](./Foundations_of_Signal_Processing_1.ipynb) Session 2 (Laplace). &nbsp; **Feeds into:** Session 3 (implementation).

---

## 3. IIR Filters

💡 **Intuition.** An IIR filter adds **feedback**: the output is a mix of recent inputs *and recent outputs*. Feedback lets a handful of coefficients ring like a resonator, so an IIR filter of order 4 can cut as sharply as an FIR of order 100. The price: the ringing must die out — every pole must sit **inside the unit circle** — and phase is no longer linear.

### 3.1. The Classical Families

Each classical design answers "what should the passband and stopband look like?" differently:

- **Butterworth** — maximally flat passband, gentle rolloff.
- **Chebyshev I** — ripples in the passband, buys a faster rolloff.
- **Elliptic** — ripples in both bands, fastest rolloff of all.

(All are analog prototypes mapped to digital via the *bilinear transform* — the Laplace $s$-plane from [Foundations Session 2](./Foundations_of_Signal_Processing_1.ipynb) bent onto the $z$-plane's unit circle.)

In [5]:
order, cutoff = 4, 100

plt.figure(figsize=(8, 3))
for name, (b, a) in {
    "Butterworth": signal.butter(order, cutoff, fs=fs),
    "Chebyshev I (1 dB)": signal.cheby1(order, 1, cutoff, fs=fs),
    "Elliptic (1 dB, 60 dB)": signal.ellip(order, 1, 60, cutoff, fs=fs),
}.items():
    w, h = signal.freqz(b, a, worN=2048, fs=fs)
    plt.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-8)), label=name)
plt.ylim(-100, 5); plt.legend(); plt.grid(True)
plt.title(f"Three order-{order} IIR low-passes — compare to the 101-tap FIR!")
plt.xlabel("frequency [Hz]"); plt.ylabel("magnitude [dB]")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1835677/811424511.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 3.2. Poles, Zeros & Stability

The transfer function $H(z)$ is a ratio of polynomials; its **zeros** pin the response down (notches) and its **poles** push it up (resonance). Stability = all poles strictly inside the unit circle.

In [6]:
b, a = signal.butter(order, cutoff, fs=fs)
z, p, _ = signal.tf2zpk(b, a)

theta = np.linspace(0, 2 * np.pi, 256)
plt.figure(figsize=(4.2, 4.2))
plt.plot(np.cos(theta), np.sin(theta), "k--", linewidth=0.8)   # unit circle
plt.plot(z.real, z.imag, "o", label="zeros")
plt.plot(p.real, p.imag, "x", markersize=9, label="poles")
plt.axis("equal"); plt.grid(True); plt.legend()
plt.title(f"Butterworth order {order}: all poles inside ⇒ stable\nmax |pole| = {np.abs(p).max():.3f}")
plt.tight_layout(); plt.show()

assert np.abs(p).max() < 1, "unstable filter!"

/tmp/ipykernel_1835677/2446283897.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Filters in Practice* (~40 min)
**Goal:** clean a real (synthetic) signal end-to-end; avoid the classic implementation pitfalls.
**Builds on:** Sessions 1–2. &nbsp; **Feeds into:** [Adaptive Filtering](../Intro_Time_Series/README.md) — filters that tune themselves.

---

## 4. Filters in Practice

### 4.1. The Scenario

An ECG-like signal contaminated by two enemies: 60 Hz power-line hum and broadband noise. Plan: a **notch** for the hum, a **low-pass** for the hiss.

In [7]:
rng = np.random.default_rng(7)
t = np.arange(0, 4, 1 / fs)

# synthetic "ECG": periodic sharp pulses + baseline wander
heart = signal.gausspulse((t % 1.0) - 0.5, fc=8) * 1.2
baseline = 0.15 * np.sin(2 * np.pi * 0.3 * t)
clean = heart + baseline

hum = 0.5 * np.sin(2 * np.pi * 60 * t)
noise = 0.15 * rng.standard_normal(t.size)
measured = clean + hum + noise

plt.figure(figsize=(9, 2.6))
plt.plot(t, measured, alpha=0.7, label="measured")
plt.plot(t, clean, linewidth=1.5, label="truth")
plt.xlim(0, 2); plt.legend(); plt.xlabel("time [s]")
plt.title("What the sensor gives you vs what you want")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1835677/539182449.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.2. Notch + Low-pass

In [8]:
# 60 Hz notch (narrow IIR band-stop)
b_notch, a_notch = signal.iirnotch(60, Q=30, fs=fs)

# 40 Hz FIR low-pass for the broadband noise
taps_lp = signal.firwin(201, 40, fs=fs)

stage1 = signal.filtfilt(b_notch, a_notch, measured)
cleaned = signal.filtfilt(taps_lp, 1, stage1)

plt.figure(figsize=(9, 2.6))
plt.plot(t, cleaned, label="filtered")
plt.plot(t, clean, "--", label="truth")
plt.xlim(0, 2); plt.legend(); plt.xlabel("time [s]")
plt.title("Notch + low-pass recovers the waveform")
plt.tight_layout(); plt.show()

rmse_before = np.sqrt(np.mean((measured - clean) ** 2))
rmse_after = np.sqrt(np.mean((cleaned - clean) ** 2))
print(f"RMSE before: {rmse_before:.3f}   after: {rmse_after:.3f}")

RMSE before: 0.385   after: 0.043


/tmp/ipykernel_1835677/2968534475.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.3. Pitfalls Worth Their Own Slide

- **`lfilter` vs `filtfilt`** — `lfilter` runs causally (real-time capable) but delays and phase-distorts; `filtfilt` runs forward *and* backward, canceling all phase distortion — offline only, since it needs the future.
- **Edge transients** — a filter needs time to "warm up"; distrust the first ~one filter length of output.
- **Numerical fragility** — high-order IIR filters in `(b, a)` form can explode from rounding. Use **second-order sections**: factor the filter into biquads with `output="sos"` + `sosfiltfilt`.

In [9]:
# The professional habit: SOS form for anything beyond order ~4
sos = signal.butter(10, 40, fs=fs, output="sos")
cleaned_sos = signal.sosfiltfilt(sos, stage1)
print("max |sos − fir| on this signal:", np.max(np.abs(cleaned_sos - cleaned)).round(3))
# small differences: two different filters approximating the same job

max |sos − fir| on this signal: 0.016


## 5. Conclusion

Design recipe to take home:

1. Waveform shape matters / offline? → **FIR** (+ `filtfilt`), taps set by transition width.
2. Compute-constrained / real-time? → **IIR** in **SOS** form, check the poles.
3. Single-frequency interference? → **notch**.
4. Always *look* at magnitude **and** phase before trusting a filter.

---
## Where next

- [Adaptive Filtering (APA / Kalman)](../Intro_Time_Series/README.md) — when the interference won't hold still, the filter must learn.
- [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb) Session 6 — how convolution with long filters is computed fast.
- [Intro to FPGA](../Intro_FPGA/README.md) — implementing these filters as literal hardware.